# Khảo Sát Dữ Liệu Thô — batdongsan.com
Xem dữ liệu **raw** vừa scrape về: hình thù, tròn méo, thiếu thốn thế nào — trước khi đưa vào pipeline.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

RAW = '../data/realestate_raw.csv'
df = pd.read_csv(RAW)
print(f"Loaded: {len(df):,} rows  ×  {df.shape[1]} cols")

**Nhận xét:** Đọc thẳng CSV thô vào pandas để xem nhanh. Số dòng × cột là kích thước tập dữ liệu thu về sau scraping — là baseline để đánh giá pipeline sau này drop bao nhiêu %.

In [ ]:
df.head(10)

**Nhận xét:** 10 dòng đầu để mắt thường kiểm tra cấu trúc: tên cột có đúng không, giá trị có bị lệch cột không, `None`/`NaN` xuất hiện ngay từ mẫu nhỏ ở đâu.

In [ ]:
print(df.dtypes.to_string())
print(f"\nTotal columns: {df.shape[1]}")

**Nhận xét:** Kiểu dữ liệu pandas infer được. Cần chú ý:
- `price_billion`, `area_m2` → phải là `float64`, nếu thấy `object` tức là cột còn dính chuỗi (đơn vị, dấu phẩy)
- `bedrooms`, `floors` → phải là `int64` hoặc `float64`
- Cột `object` không phải text thuần → dấu hiệu cần parse thêm trong cleaning

In [ ]:

# --- Bảng null thô: xem thẳng số liệu trước khi nhìn chart ---
miss       = df.isnull().sum().sort_values(ascending=False)
miss_pct   = (miss / len(df) * 100).round(2)
miss_table = pd.DataFrame({
    'null_count' : miss,
    'null_%'     : miss_pct,
    'non_null'   : len(df) - miss,
    'dtype'      : df.dtypes,
})

# highlight hàng có null
def color_null(val):
    if isinstance(val, float):
        if val > 30:  return 'background-color: #f1948a'
        if val > 5:   return 'background-color: #f9e79f'
    return ''

miss_table.style.applymap(color_null, subset=['null_%']).format({'null_%': '{:.2f}%'})


**Nhận xét:** Bảng null đầy đủ: `null_count` (số dòng thiếu), `null_%` (tỷ lệ), `non_null` (số dòng có giá trị), `dtype` (kiểu). Màu **đỏ** = thiếu > 30%, **vàng** = thiếu 5–30% — nhìn ngay ra cột nào cần ưu tiên xử lý.

In [ ]:
miss = df.isnull().sum().sort_values(ascending=False)
miss_pct = miss / len(df) * 100
miss_df = pd.DataFrame({'count': miss, 'pct': miss_pct})

fig, ax = plt.subplots(figsize=(10, max(4, len(miss_df) * 0.35)))
colors = ['#e74c3c' if p > 30 else '#f39c12' if p > 5 else '#2ecc71' for p in miss_df['pct']]
bars = ax.barh(miss_df.index[::-1], miss_df['pct'][::-1], color=colors[::-1])
for bar, (_, row) in zip(bars, miss_df[::-1].iterrows()):
    if row['count'] > 0:
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{row['pct']:.1f}%  ({int(row['count']):,})",
                va='center', fontsize=8.5)
ax.axvline(30, color='#e74c3c', linestyle='--', linewidth=0.8, alpha=0.6, label='30% threshold')
ax.axvline(5,  color='#f39c12', linestyle='--', linewidth=0.8, alpha=0.6, label='5% threshold')
ax.set_xlabel('% Missing')
ax.set_title('Missing Values theo cột  (đỏ > 30% | cam > 5% | xanh OK)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Nhận xét:**
- **Xanh (< 5%)** → OK, có thể drop dòng hoặc fill median/mode
- **Cam (5–30%)** → cần chiến lược impute rõ ràng trước khi dùng
- **Đỏ (> 30%)** → cột kém tin cậy, thường là `direction`, `floors`, `legal_status` vì người đăng không bắt buộc điền

Cột nào đỏ mà không cần thiết cho model → nên loại hẳn khỏi feature set.

In [ ]:
num_cols = [c for c in ['price_billion','area_m2','price_per_m2_million',
                         'bedrooms','bathrooms','floors'] if c in df.columns]
df[num_cols].describe().T.style.format('{:.2f}').background_gradient(axis=1, cmap='Blues')

**Nhận xét:** Thống kê mô tả có màu để dễ so sánh. Dấu hiệu cần lọc:
- `min` = 0 hoặc âm ở giá/diện tích → lỗi parse
- `max` cực lớn so với `75%` → outlier nặng (ví dụ giá 10.000 tỷ)
- `std` >> `mean` → phân phối lệch mạnh, median đại diện tốt hơn mean

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Giá
if 'price_billion' in df.columns:
    s = df['price_billion'].dropna()
    p95 = s.quantile(0.95)
    axes[0].hist(s[s <= p95], bins=50, color='steelblue', edgecolor='white')
    axes[0].axvline(s.median(), color='red', linestyle='--', label=f'Median {s.median():.1f} tỷ')
    axes[0].axvline(s.mean(),   color='orange', linestyle='--', label=f'Mean {s.mean():.1f} tỷ')
    axes[0].set_title(f'Phân phối Giá (≤ p95={p95:.0f} tỷ)  |  raw n={len(s):,}')
    axes[0].set_xlabel('Tỷ VND')
    axes[0].legend()

# Diện tích
if 'area_m2' in df.columns:
    s = df['area_m2'].dropna()
    p95 = s.quantile(0.95)
    axes[1].hist(s[s <= p95], bins=50, color='mediumpurple', edgecolor='white')
    axes[1].axvline(s.median(), color='red', linestyle='--', label=f'Median {s.median():.0f} m²')
    axes[1].axvline(s.mean(),   color='orange', linestyle='--', label=f'Mean {s.mean():.0f} m²')
    axes[1].set_title(f'Phân phối Diện tích (≤ p95={p95:.0f} m²)  |  raw n={len(s):,}')
    axes[1].set_xlabel('m²')
    axes[1].legend()

plt.suptitle('Phân phối 2 cột số quan trọng nhất', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Nhận xét:** Cả giá và diện tích đều phân phối **lệch phải (right-skewed)** — đuôi dài về phía cao. Khoảng cách `mean` > `median` xác nhận điều này. Cắt tại p95 để loại outlier cực đoan mà không mất dữ liệu đại diện. Pipeline cleaning sẽ cần xử lý outlier này trước khi huấn luyện model.

In [ ]:
cat_cols = [c for c in ['property_type', 'city', 'district', 'legal_status', 'direction'] if c in df.columns]
ncols = 2
nrows = (len(cat_cols) + 1) // 2
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    vc = df[col].value_counts().dropna().head(12)
    axes[i].barh(vc.index[::-1], vc.values[::-1], color='steelblue')
    for j, (val, cnt) in enumerate(zip(vc.values[::-1], vc.values[::-1])):
        axes[i].text(val + max(vc.values)*0.01, j, f'{val:,}', va='center', fontsize=8)
    n_unique = df[col].nunique()
    axes[i].set_title(f'{col}  (unique: {n_unique})', fontweight='bold')
    axes[i].set_xlabel('Số tin đăng')

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Phân bổ các cột phân loại (top 12)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Nhận xét:** Nhìn vào đây ngay thấy:
- Giá trị rác từ scraper: `'None'`, `'--'`, `'Đang cập nhật'` → cần clean
- `unique` count cao bất thường ở `district` → viết hoa/thường không nhất quán (`Quận 1` vs `quận 1`)
- Một loại BĐS chiếm áp đảo → dữ liệu mất cân bằng, ảnh hưởng model phân loại

In [ ]:
total = len(df)
id_col = 'listing_id' if 'listing_id' in df.columns else None

if id_col:
    dupes = df.duplicated(subset=[id_col]).sum()
    label = f"duplicate '{id_col}'"
else:
    dupes = df.duplicated().sum()
    label = "duplicate toàn dòng"

print(f"Tổng dòng   : {total:,}")
print(f"Duplicate   : {dupes:,}  ({dupes/total*100:.1f}%)  [{label}]")
print()

# Pie chart tỷ lệ
fig, ax = plt.subplots(figsize=(5, 4))
sizes  = [total - dupes, dupes]
labels = [f'Dữ liệu hợp lệ\n{total-dupes:,}', f'Duplicate\n{dupes:,}']
colors = ['#2ecc71', '#e74c3c']
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title('Tỷ lệ Duplicate')
plt.tight_layout()
plt.show()

**Nhận xét:** Duplicate xuất hiện do scraper chạy nhiều lần hoặc cùng một tin đăng xuất hiện ở nhiều trang kết quả. Duplicate làm sai lệch toàn bộ thống kê (count bị thổi phồng, mean bị kéo lệch). Đây là bước **bắt buộc** phải dedup trước khi đẩy vào Kafka/Hive.

In [ ]:
miss = df.isnull().sum()
miss_pct = miss / len(df) * 100

print("=" * 55)
print("  TÓM TẮT KHẢO SÁT DỮ LIỆU THÔ")
print("=" * 55)
print(f"  Tổng tin đăng       : {len(df):,}")
print(f"  Số cột              : {df.shape[1]}")
if 'district' in df.columns:
    print(f"  Quận/huyện unique   : {df['district'].nunique()}")
if 'city' in df.columns:
    print(f"  Thành phố unique    : {df['city'].nunique()}")
if 'property_type' in df.columns:
    print(f"  Loại BĐS unique     : {df['property_type'].nunique()}")
print()
bad_cols  = miss_pct[miss_pct > 30].index.tolist()
warn_cols = miss_pct[(miss_pct > 5) & (miss_pct <= 30)].index.tolist()
print(f"  Cột missing > 30%   : {bad_cols  if bad_cols  else '✅ Không có'}")
print(f"  Cột missing 5–30%   : {warn_cols if warn_cols else '✅ Không có'}")
print(f"  Duplicate rows      : {dupes:,}  ({dupes/len(df)*100:.1f}%)")
print("=" * 55)

**Nhận xét:** Snapshot tổng kết để báo cáo nhanh — dữ liệu thô thu về quy mô bao nhiêu, vấn đề chính là gì. Từ đây xác định ưu tiên bước cleaning: dedup → parse kiểu dữ liệu → xử lý missing → lọc outlier → đẩy vào pipeline.